# Repo Analyzer MVP — Colab smoke test на реальной Qwen2.5-Coder 7B Q4

Этот ноутбук проверяет **PR #28**: `GitHub URL → explorer tool calls → read_file → изолированный extractor → RepoEvidence → analysis.json / report.md`.

**Запуск:** Google Colab → *Runtime → Change runtime type → T4 GPU* → выполняй ячейки сверху вниз. Нужны доступ в интернет и место под GGUF (~5 GB). Применяются код агента из feature-ветки и тот же Q4-файл, что использовался в benchmark v8.

**Важно:** это smoke test на реальной модели, **не** проверка semantic grounding и не benchmark качества. Успешное завершение CLI (код 0) само по себе не означает, что агент нашёл хорошие факты: проверь `status`, `tool_trace`, `evidence` и доказательства по ссылкам. Не загружай приватные репозитории без отдельной проверки безопасности исходников.

## 1. Настройки и GPU

Для первого запуска оставь публичный `military-charters-rag`. Модель и сервер находятся **в этом Colab runtime**, без удалённого LLM API. При необходимости укажи `GITHUB_TOKEN` в Colab Secrets; токен не требуется для одного небольшого публичного репозитория.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

TARGET_REPO = "Ferraronp/military-charters-rag"  # @param {type:"string"}
MAX_STEPS = 10  # @param {type:"integer"}
MAX_READS = 4  # @param {type:"integer"}

PR_BRANCH = "feat/deep-repo-exploration"
REPO_URL = "https://github.com/Ferraronp/github2CV_ML.git"
REPO_DIR = Path("/content/github2cv_mvp_pr28")
WORKDIR = Path("/content/github2cv_deep_exploration_smoke")
WORKDIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR = WORKDIR / "results"

MODEL_REPO = "unsloth/Qwen2.5-Coder-7B-Instruct-GGUF"
MODEL_FILE = "Qwen2.5-Coder-7B-Instruct-Q4_K_M.gguf"
MODEL_ALIAS = "qwen-coder"

assert 1 <= MAX_STEPS <= 50
assert 1 <= MAX_READS <= MAX_STEPS
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
     "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
)
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is not None:
    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None
    if github_token:
        os.environ["GITHUB_TOKEN"] = github_token
print("GPU:", gpu.stdout.strip())
print("Repo to inspect:", TARGET_REPO)
print("Explorer budget:", MAX_STEPS, "steps /", MAX_READS, "reads")

## 2. Установка текущей ветки MVP и Python-зависимостей

Здесь не выполняется архивный Colab v8, а устанавливается **именно приложение из PR #28**.

In [ ]:
if REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch",
         "--branch", PR_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

active_branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True
).strip()
assert active_branch == PR_BRANCH, f"Expected {PR_BRANCH}, got {active_branch}"
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Agent source commit:", commit)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "huggingface_hub", "nvidia-cuda-runtime-cu12==12.8.90",
     "nvidia-cublas-cu12==12.8.5.5"],
    check=True,
)
print("MVP package and CUDA runtime wheels installed.")

## 3. CUDA-сборка `llama-server`

Используется архив **ai-dock b8660 / CUDA 12.8**, как в эксперименте v8, с закреплённой SHA-256. Для этой готовой сборки необходим совместимый драйвер NVIDIA (в исходном бенчмарке проверяли версию **570.15+**). Если окружение Colab отличается, ошибка выводится явно, а не маскируется под неудачу Qwen.

In [ ]:
import hashlib
import importlib.metadata as package_metadata

PREBUILT_URL = (
    "https://github.com/ai-dock/llama.cpp-cuda/releases/download/"
    "b8660/llama.cpp-b8660-cuda-12.8.tar.gz"
)
PREBUILT_SHA256 = (
    "7accea053a9e9ccfae22825fa975a873129e51ef0220b59bed3aee703823ffae"
)
SERVER_HOME = WORKDIR / "server_b8660"
SERVER_HOME.mkdir(exist_ok=True)
ARCHIVE = SERVER_HOME / "llama.cpp-b8660-cuda-12.8.tar.gz"
SERVER_BIN = SERVER_HOME / "cuda-12.8" / "llama-server"

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

driver = gpu.stdout.split(",")[1].strip()
parts = tuple(int(part) for part in driver.split(".")[:2])
assert parts >= (570, 15), f"CUDA 12.8 prebuilt requires driver >=570.15; got {driver}"

if not ARCHIVE.exists() or sha256_file(ARCHIVE) != PREBUILT_SHA256:
    temp_archive = ARCHIVE.with_suffix(".part")
    subprocess.run(
        ["wget", "-q", "--show-progress", "--tries=3", "-L",
         PREBUILT_URL, "-O", str(temp_archive)],
        check=True,
    )
    temp_archive.replace(ARCHIVE)
assert sha256_file(ARCHIVE) == PREBUILT_SHA256, "Prebuilt checksum mismatch"
if not SERVER_BIN.exists():
    subprocess.run(
        ["tar", "-xzf", str(ARCHIVE), "-C", str(SERVER_HOME)],
        check=True,
    )
assert SERVER_BIN.is_file(), "llama-server binary not found in archive"
SERVER_BIN.chmod(SERVER_BIN.stat().st_mode | 0o111)

def cuda_library_directory(distribution, suffix):
    package = package_metadata.distribution(distribution)
    for entry in package.files or []:
        if str(entry).endswith(suffix):
            full = Path(package.locate_file(entry))
            if full.exists():
                return str(full.parent)
    raise RuntimeError(f"Missing library {suffix} in {distribution}")

runtime_lib = cuda_library_directory(
    "nvidia-cuda-runtime-cu12", "nvidia/cuda_runtime/lib/libcudart.so.12"
)
cublas_lib = cuda_library_directory(
    "nvidia-cublas-cu12", "nvidia/cublas/lib/libcublas.so.12"
)
SERVER_ENV = os.environ.copy()
SERVER_ENV["LD_LIBRARY_PATH"] = os.pathsep.join(
    [str(SERVER_BIN.parent), runtime_lib, cublas_lib,
     SERVER_ENV.get("LD_LIBRARY_PATH", "")]
)
linked = subprocess.run(
    ["ldd", str(SERVER_BIN)], env=SERVER_ENV,
    check=True, capture_output=True, text=True,
)
missing = [line for line in linked.stdout.splitlines() if "not found" in line]
assert not missing, "Missing shared libraries: " + "; ".join(missing)
print("llama-server binary and CUDA libraries verified:", SERVER_BIN)

## 4. Загрузка GGUF

Здесь загружается **Qwen2.5-Coder-7B-Instruct Q4_K_M** из Unsloth на Hugging Face, та же конфигурация, которая дала лучший предварительный Grounded F1 в нашем v8. Скачивание модели может быть крупным.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

MODEL_DIR = WORKDIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
model_info = HfApi().model_info(MODEL_REPO)
model_revision = model_info.sha
available_files = {file.rfilename for file in model_info.siblings}
assert MODEL_FILE in available_files, f"GGUF not available: {MODEL_FILE}"
MODEL_PATH = Path(hf_hub_download(
    repo_id=MODEL_REPO, filename=MODEL_FILE,
    revision=model_revision, local_dir=MODEL_DIR
))
print("Hugging Face model revision:", model_revision)
print("GGUF:", MODEL_PATH)
print("GGUF size (GiB):", round(MODEL_PATH.stat().st_size / (1024 ** 3), 2))

## 5. Запуск локального сервера с tool calling

Критичен флаг `--jinja`, а имя модели должно совпадать с `--alias qwen-coder`. Сервер слушает только `127.0.0.1` внутри Colab. **Порт выбирается автоматически**, поэтому занятый `8080` не мешает запуску. Выбранный адрес используется для `/health`, проверки `tool_calls` и запуска CLI. Если запуск всё же завершится ошибкой, будет показан хвост лога.

In [ ]:
import socket
import time
from urllib.error import URLError
from urllib.request import urlopen

# A previous Colab cell may already have a server bound to port 8080.
# Stop only this notebook's earlier process; never kill unrelated servers.
previous_process = globals().get("SERVER_PROCESS")
if previous_process is not None and previous_process.poll() is None:
    previous_process.terminate()
    try:
        previous_process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        previous_process.kill()
        previous_process.wait(timeout=10)
previous_log = globals().get("SERVER_LOG_HANDLE")
if previous_log is not None and not previous_log.closed:
    previous_log.close()

# Request a free loopback port from the OS, rather than assuming 8080.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    SERVER_PORT = port_probe.getsockname()[1]
SERVER_BASE_URL = f"http://127.0.0.1:{SERVER_PORT}/v1"
SERVER_HEALTH_URL = f"http://127.0.0.1:{SERVER_PORT}/health"
print("Selected free loopback port:", SERVER_PORT)
SERVER_LOG_PATH = WORKDIR / "llama-server.log"
SERVER_LOG_HANDLE = SERVER_LOG_PATH.open("w", encoding="utf-8")
SERVER_PROCESS = subprocess.Popen(
    [
        str(SERVER_BIN),
        "--model", str(MODEL_PATH),
        "--alias", MODEL_ALIAS,
        "--jinja",
        "--ctx-size", "8192",
        "--n-gpu-layers", "99",
        "--parallel", "1",
        "--host", "127.0.0.1",
        "--port", str(SERVER_PORT),
    ],
    stdout=SERVER_LOG_HANDLE,
    stderr=subprocess.STDOUT,
    env=SERVER_ENV,
)
print("llama-server PID:", SERVER_PROCESS.pid)
ready = False
deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    if SERVER_PROCESS.poll() is not None:
        break
    try:
        with urlopen(SERVER_HEALTH_URL, timeout=3) as response:
            if response.status == 200:
                ready = True
                break
    except (URLError, TimeoutError, OSError):
        pass
    time.sleep(2)

if not ready:
    SERVER_LOG_HANDLE.flush()
    tail = SERVER_LOG_PATH.read_text(errors="replace").splitlines()[-35:]
    print("Server log (last lines):\n" + "\n".join(tail))
    raise RuntimeError("llama-server failed to become healthy")
print("llama-server healthy on", SERVER_BASE_URL)

## 6. Диагностика tool calling (отдельно от агента)

Короткая проверка, возвращает ли сама Qwen структурированный `tool_calls` через наш `LocalOpenAIModel`. Если модель вместо инструмента пишет текст, это не считается пройденной проверкой — такой результат важно сохранить для отладки шаблона/версии llama.cpp.

In [ ]:
import sys as system_module

if str(REPO_DIR / "src") not in system_module.path:
    system_module.path.insert(0, str(REPO_DIR / "src"))

from github2cv_ml.agent.runner import LocalOpenAIModel

model = LocalOpenAIModel(
    base_url=SERVER_BASE_URL, model=MODEL_ALIAS, timeout=180
)
diagnostic_tool = [{
    "type": "function",
    "function": {
        "name": "echo_probe",
        "description": "Return the provided text without modification",
        "parameters": {
            "type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"],
        },
    },
}]
probe = model.complete(
    [
        {"role": "system", "content": "Call echo_probe once. Do not answer in prose."},
        {"role": "user", "content": "Call echo_probe with text ping."},
    ],
    tools=diagnostic_tool,
)
probe_calls = probe.get("tool_calls") if isinstance(probe, dict) else None
print("Structured tool call (required mode):", bool(probe_calls))
if probe_calls:
    print("Tool name:", probe_calls[0].get("function", {}).get("name"))
    print("Arguments:", probe_calls[0].get("function", {}).get("arguments"))
else:
    print("WARNING: Model did not produce tool_calls.")
    print("Response:", str(probe.get("content") if isinstance(probe, dict) else probe)[:800])
    print("The model did not call a tool even with tool_choice=required.")
    print("Check --jinja, chat template, and llama-server version before trusting agent output.")

## 7. Реальный end-to-end запуск Repo Analyzer

Вызываем **настоящую CLI-команду**, которая поставляется в PR #28. Изменив `TARGET_REPO`, можно тестировать другие публичные репозитории. Параметры бюджета контролируют число tool calls и количество прочитанных фрагментов. Если endpoint не работает, приложение должно сохранить неполный отчёт, а не потерять уже собранные данные.

In [ ]:
import json

cmd = [
    sys.executable, "-m", "github2cv_ml.agent", TARGET_REPO,
    "--base-url", SERVER_BASE_URL,
    "--model", MODEL_ALIAS,
    "--max-steps", str(MAX_STEPS),
    "--max-reads", str(MAX_READS),
    "--output", str(RESULT_DIR),
]
started = time.monotonic()
run = subprocess.run(
    cmd, cwd=str(REPO_DIR), env=os.environ.copy(),
    text=True, capture_output=True, timeout=2400,
)
elapsed = time.monotonic() - started
print("CLI exit code:", run.returncode)
print("Runtime (seconds):", round(elapsed, 1))
print("stdout:\n" + run.stdout[-4000:])
if run.stderr:
    print("stderr:\n" + run.stderr[-5000:])
if run.returncode != 0:
    raise RuntimeError("Repo Analyzer CLI failed; see stderr above")
if not (RESULT_DIR / "analysis.json").exists():
    raise RuntimeError("CLI finished but analysis.json is absent")
analysis = json.loads((RESULT_DIR / "analysis.json").read_text(encoding="utf-8"))
print("Run status:", analysis["status"])
print("Evidence count:", len(analysis["evidence"]))
print("Tool actions:", len(analysis["tool_trace"]))
print("Selected-file coverage:", analysis.get("coverage", {}).get("planned_coverage_ratio"))

## 8. Проверка результата и источников

Здесь видно, **что именно делала модель**: какие инструменты вызывала, сколько было успешных чтений и что извлекла. Открой ссылки на код и вручную проверь смысл первых 3–5 claims. `verification_level=locator_only` означает, что автоматической семантической проверки здесь нет.

In [ ]:
from IPython.display import Markdown, display

trace = analysis["tool_trace"]
read_ok = sum(
    action.get("tool") == "read_file" and action.get("status") == "ok"
    for action in trace
)
errors = sum(action.get("status") == "error" for action in trace)
completed = analysis["status"] == "completed"
has_evidence = bool(analysis["evidence"])
print("Repository:", analysis["repository"]["url"])
print("Pinned commit SHA:", analysis["commit_sha"])
print("Status:", analysis["status"])
print("Verification level:", analysis["verification_level"])
print("Read calls (successful):", read_ok)
print("Tool errors:", errors)
print("Warnings:", analysis["warnings"])
print("Source-map plan (filename hypotheses):")
for item in analysis.get("exploration_plan", []):
    print(" ", item["candidate_role"], item["path"])
print("Coverage of selected targets (not whole repository):")
print(json.dumps(analysis.get("coverage", {}), indent=2, ensure_ascii=False))
print()
print("Exploration trace:")
for action in trace:
    print(
        "step", action.get("step"), "|", action.get("tool"),
        "|", action.get("status"), "|", action.get("arguments", {})
    )

print("\nExtracted evidence:")
for evidence in analysis["evidence"][:10]:
    source = evidence["source"]
    print(
        f"- {evidence['observation']}\n"
        f"  {source['path']}:{source['line_start']}-{source['line_end']}\n"
        f"  {source['url']}"
    )
print("\nManual audit: verify the first 3–5 evidence URLs against the claims.")
if completed and read_ok > 0 and has_evidence:
    print("SMOKE RESULT: completed + used read_file + produced locator-backed claims.")
else:
    print("SMOKE RESULT: incomplete / no successful reads / no observations.")
    print("Inspect llama-server.log and tool_trace before drawing conclusions.")
display(Markdown((RESULT_DIR / "report.md").read_text(encoding="utf-8")))

## 9. Сохранить результаты

Скачиваем только итоговые `analysis.json` и `report.md`. Они могут включать извлечённые сведения о репозитории и пути к файлам; не публикуй артефакты анализа приватных проектов без проверки. Лог `llama-server` остаётся в Colab и доступен для отладки.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

bundle_path = WORKDIR / "repo_analyzer_smoke_results.zip"
with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as bundle:
    for filename in ("analysis.json", "report.md"):
        result_file = RESULT_DIR / filename
        if result_file.exists():
            bundle.write(result_file, arcname=filename)
print("Results archive:", bundle_path)
try:
    from google.colab import files

    files.download(str(bundle_path))
except ImportError:
    print("Not in Colab; open the ZIP at the location above.")

## 10. Остановить сервер (необязательно)

После проверки освобождает VRAM. Ячейку можно выполнять только после завершения анализа.

In [ ]:
if "SERVER_PROCESS" in globals() and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    try:
        SERVER_PROCESS.wait(timeout=20)
    except subprocess.TimeoutExpired:
        SERVER_PROCESS.kill()
        SERVER_PROCESS.wait(timeout=10)
if "SERVER_LOG_HANDLE" in globals():
    SERVER_LOG_HANDLE.close()
print("llama-server stopped (if it was running).")